# 01 — EDA inicial

Objetivo: **olhar os dados antes de decidir qualquer arquitetura**. Este notebook só descreve, não decide nada.

Pipeline-alvo: tratamento (heurístico) → CNN1 *tem medidor?* → CNN2 *digital ou analógico?* → CNN3 *localizador de campos* → CNN4 *OCR*.

In [ ]:
import glob
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

# Raiz dos dados. Pode ser trocada com a variável de ambiente LEITURISTA_DATA.
DATA = Path(os.environ.get("LEITURISTA_DATA", Path.home() / "Developer/leiturista/data"))
print(DATA, "existe:", DATA.exists())

## 1. Fotos de campo da distribuidora

São 4 lotes diários. Cada lote é uma pasta com um CSV (separador `;`) e os JPGs.

### 1.1 Juntar os 4 CSVs numa tabela só

In [ ]:
tabelas = []
for arquivo_csv in sorted(DATA.glob("distribuidora_campo/*/*.csv")):
    pasta = arquivo_csv.parent
    tabela = pd.read_csv(arquivo_csv, sep=";", dtype=str)
    tabela["lote"] = pasta.name[-11:]   # ex.: 030726_0121
    tabela["pasta"] = str(pasta)
    tabelas.append(tabela)

fotos = pd.concat(tabelas, ignore_index=True)

# Nomes de coluna originais são longos; renomeio pra facilitar.
fotos.columns = ["medidor", "leitura", "nota", "foto", "lote", "pasta"]
fotos.head()

### 1.2 Quais linhas têm foto de verdade no disco?

In [ ]:
def foto_existe(linha):
    # Algumas linhas do CSV vêm sem nome de foto (NaN).
    if pd.isna(linha.foto):
        return False
    return os.path.exists(f"{linha.pasta}/{linha.foto}")

fotos["caminho"] = fotos.pasta + "/" + fotos.foto.fillna("")
fotos["tem_foto"] = fotos.apply(foto_existe, axis=1)

print(f"linhas no CSV:            {len(fotos)}")
print(f"linhas sem nome de foto:  {fotos.foto.isna().sum()}")
print(f"linhas com foto no disco: {fotos.tem_foto.sum()}")
print(f"JPGs no disco:            {len(glob.glob(str(DATA / 'distribuidora_campo/*/*.jpg')))}")

fotos.groupby("lote").agg(linhas=("foto", "size"), com_foto=("tem_foto", "sum"))

Se os JPGs no disco forem mais que as linhas com foto, sobram fotos **órfãs** (existem no disco mas nenhum CSV aponta pra elas).

### 1.3 Distribuição das notas do leiturista

In [ ]:
notas = fotos.nota.fillna("(vazio)").value_counts()
print(len(notas), "notas distintas")

notas.head(20).plot.barh(figsize=(6, 5)).invert_yaxis()
plt.title("20 notas mais frequentes")
plt.show()

### 1.4 Tamanho, brilho e contraste

Medido numa amostra de 600 fotos (abrir todas seria lento e não muda o quadro geral).

In [ ]:
def medidas_da_imagem(caminho):
    imagem = Image.open(caminho)
    largura, altura = imagem.size
    orientacao_exif = imagem.getexif().get(274)   # tag 274 = Orientation

    # Reduzo pra 128x128 em tons de cinza só pra medir brilho/contraste rápido.
    cinza = imagem.convert("L").resize((128, 128))
    pixels = np.asarray(cinza, dtype=float)
    return largura, altura, pixels.mean(), pixels.std(), orientacao_exif


amostra = fotos[fotos.tem_foto].sample(600, random_state=0).copy()
medidas = [medidas_da_imagem(c) for c in amostra.caminho]
amostra[["largura", "altura", "brilho", "contraste", "exif"]] = medidas

print("tamanhos (largura x altura) mais comuns:")
print(amostra.groupby(["largura", "altura"]).size().sort_values(ascending=False).head())
print("\norientação EXIF:", amostra.exif.value_counts(dropna=False).to_dict())

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(12, 3))

amostra.brilho.hist(ax=eixos[0])
eixos[0].set_title("brilho médio (0-255)")

amostra.contraste.hist(ax=eixos[1])
eixos[1].set_title("contraste (desvio padrão)")

(amostra.largura / amostra.altura).hist(ax=eixos[2])
eixos[2].set_title("proporção largura/altura")

plt.show()

### 1.5 Olhar as fotos

Estatística não substitui ver. Primeiro 16 fotos aleatórias (título = nota), depois as 4 notas mais comuns.

In [ ]:
def mostrar_fotos(caminhos, titulos=None, colunas=4, lado=256):
    caminhos = list(caminhos)
    linhas = -(-len(caminhos) // colunas)   # divisão pra cima
    fig, eixos = plt.subplots(linhas, colunas, figsize=(colunas * 3, linhas * 3))
    eixos = np.ravel(eixos)

    for eixo in eixos:
        eixo.axis("off")

    for i, caminho in enumerate(caminhos):
        imagem = ImageOps.exif_transpose(Image.open(caminho)).convert("RGB")
        imagem.thumbnail((lado, lado))
        eixos[i].imshow(imagem)
        if titulos is not None:
            eixos[i].set_title(str(titulos[i]), fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
sorteio = fotos[fotos.tem_foto].sample(16, random_state=1)
mostrar_fotos(sorteio.caminho, sorteio.nota.fillna("-").tolist())

In [ ]:
# A nota é um rótulo indireto: aqui só checo se cena e nota parecem se relacionar.
for nota in notas.index[:4]:
    da_nota = fotos[(fotos.nota.fillna("(vazio)") == nota) & fotos.tem_foto]
    print(f"nota {nota}: {len(da_nota)} fotos")
    mostrar_fotos(da_nota.sample(8, random_state=2).caminho)

## 2. UFPR-AMR (crops de display com a leitura)

Já vêm recortados no display do medidor. Servem pro OCR (CNN4), não pros classificadores de cena.

In [ ]:
ufpr = pd.read_csv(DATA / "finetune_ufpramr/labels.csv", dtype={"label": str})
ufpr["caminho"] = str(DATA / "finetune_ufpramr") + "/" + ufpr.image
ufpr["n_digitos"] = ufpr.label.str.len()

tamanhos = [Image.open(c).size for c in ufpr.caminho]
ufpr[["largura", "altura"]] = tamanhos

print("por split:", ufpr.split.value_counts().to_dict())
print("nº de dígitos por leitura:", ufpr.n_digitos.value_counts().sort_index().to_dict())
print(ufpr[["largura", "altura"]].describe().loc[["min", "50%", "max"]])

In [ ]:
todos_os_digitos = pd.Series(list("".join(ufpr.label)))
todos_os_digitos.value_counts().sort_index().plot.bar(figsize=(5, 2.5), title="frequência de cada dígito")
plt.show()

exemplos = ufpr.sample(12, random_state=0)
mostrar_fotos(exemplos.caminho, exemplos.label.tolist())